# 🚀 TrafficVision-Transformer — Train STE-TTE (Người 2)

**Notebook này dành cho Người 2.** Chạy từng ô theo thứ tự, đọc hướng dẫn trước mỗi ô.

> ⚠️ Trước khi chạy: **Runtime → Change runtime type → T4 GPU**

## BƯỚC 1 — Kiểm tra GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU!')
!nvidia-smi

CUDA available: True
GPU: Tesla T4
Tue Jun  2 15:28:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------

## BƯỚC 2 — Mount Google Drive

Sẽ hiện popup yêu cầu đăng nhập Google. Nhấn **Connect to Google Drive**.

In [2]:
from google.colab import drive
drive.mount('/content/drive/')
print('Drive mounted OK!')

Mounted at /content/drive/
Drive mounted OK!


## BƯỚC 3 — Clone code từ GitHub

**Sửa `YOUR_GITHUB_REPO_URL` thành link repo của nhóm.**

Ví dụ: `https://github.com/ten-nhom/TrafficVision-Transformer.git`

Nếu repo **private**, dùng token:
`https://<TOKEN>@github.com/ten-nhom/TrafficVision-Transformer.git`

In [3]:

import os

GITHUB_REPO_URL = 'https://github.com/MinhCYB/TrafficVision-Transformer.git'
REPO_DIR = '/content/TrafficVision-Transformer'

if os.path.exists(REPO_DIR):
    print('Repo đã tồn tại, pulling bản mới nhất...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {GITHUB_REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

print('Working dir:', os.getcwd())

Cloning into '/content/TrafficVision-Transformer'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 177 (delta 94), reused 112 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 374.24 KiB | 24.95 MiB/s, done.
Resolving deltas: 100% (94/94), done.
Working dir: /content/TrafficVision-Transformer


## BƯỚC 4 — Cài dependencies

Colab đã có sẵn torch/torchvision, chỉ cần cài thêm `timm`.

In [4]:
!pip install timm -q

## BƯỚC 5 — Trỏ đến dataset UIT-ADrone

Dataset phải được upload lên Google Drive theo cấu trúc:
```
My Drive/
└── UIT-ADrone/
    ├── train/
    │   └── frames/   ← chứa các folder scene với .jpg
    └── test/
        ├── frames/   ← chứa các folder scene với .jpg
        └── test_frame_mask/  ← chứa các file .npy label
```

**Chạy ô này để kiểm tra dataset có đúng vị trí không.**

In [5]:

import os

DATA_DIR = "/content/drive/MyDrive/seminar/UIT-ADrone"

print(os.listdir(DATA_DIR))
print()

print("TRAIN:")
print(os.listdir(os.path.join(DATA_DIR, "train"))[:10])

print()

print("TEST:")
print(os.listdir(os.path.join(DATA_DIR, "test"))[:10])

['test.json', 'train.json', 'README.txt', 'videos', 'test', 'train', 'snippets']

TRAIN:
['frames']

TEST:
['test_frame_mask', 'frames']


In [6]:
import os

# ← SỬA đường dẫn nếu bạn để dataset ở thư mục khác trong Drive
DATA_DIR = '/content/drive/MyDrive/seminar/UIT-ADrone'

# Kiểm tra cấu trúc
train_path = os.path.join(DATA_DIR, 'train/frames')
test_path  = os.path.join(DATA_DIR, 'test/frames')
label_path = os.path.join(DATA_DIR, 'test/test_frame_mask')

print('Train frames:', os.path.exists(train_path), '→', train_path)
print('Test frames: ', os.path.exists(test_path),  '→', test_path)
print('Labels:      ', os.path.exists(label_path), '→', label_path)

if os.path.exists(train_path):
    scenes = os.listdir(train_path)
    print(f'\nTìm thấy {len(scenes)} scene trong train: {scenes[:5]}...')
else:
    print('\n❌ KHÔNG TÌM THẤY DATASET! Kiểm tra lại đường dẫn DATA_DIR.')

Train frames: True → /content/drive/MyDrive/seminar/UIT-ADrone/train/frames
Test frames:  True → /content/drive/MyDrive/seminar/UIT-ADrone/test/frames
Labels:       True → /content/drive/MyDrive/seminar/UIT-ADrone/test/test_frame_mask

Tìm thấy 41 scene trong train: ['50m_90d_morning_congkhuA_22_3', 'DJI_0070', '50m_90d_morning_ngatuanninh_30_3', '70m_90d_morning_ngatuanninh_4_3', 'DJI_0067']...


In [7]:
import os

for root, dirs, files in os.walk(DATA_DIR):
    print(root)
    print("dirs:", dirs[:3])
    print("files:", files[:3])
    print("-"*50)

    # tránh in quá nhiều
    break

/content/drive/MyDrive/seminar/UIT-ADrone
dirs: ['videos', 'test', 'train']
files: ['test.json', 'train.json', 'README.txt']
--------------------------------------------------


## BƯỚC 6 — Tạo thư mục lưu checkpoint trên Drive

Checkpoint sẽ được lưu thẳng vào Drive để không mất khi Colab disconnect.

In [8]:
!mkdir -p /content/drive/MyDrive/model_save

In [9]:
!mkdir -p /content/TrafficVision-Transformer/experiments_andt_ADrone_STE_TTE

!rm -rf /content/TrafficVision-Transformer/experiments_andt_ADrone_STE_TTE/checkpoints

!ln -s /content/drive/MyDrive/model_save \
/content/TrafficVision-Transformer/experiments_andt_ADrone_STE_TTE/checkpoints

In [10]:
!ls -l /content/TrafficVision-Transformer/experiments_andt_ADrone_STE_TTE

total 0
lrwxrwxrwx 1 root root 33 Jun  2 15:28 checkpoints -> /content/drive/MyDrive/model_save


## BƯỚC 7 — TRAINING 🚂

Chạy ô này để bắt đầu train. Log sẽ hiện trực tiếp bên dưới.

**Ước tính thời gian:** ~30-90 phút tùy GPU Colab cấp.

> ⚠️ **Đừng đóng tab** trong lúc chạy! Nếu bị disconnect thì checkpoint vẫn còn trên Drive.

In [11]:
import os

DATA_DIR = "/content/drive/MyDrive/seminar/UIT-ADrone"

print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print()

for root, dirs, files in os.walk(DATA_DIR):
    print("ROOT:", root)
    print("DIRS:", dirs[:5])
    print("FILES:", files[:5])
    print("-"*50)

    # tránh spam
    if root.count(os.sep) > 3:
        break

DATA_DIR exists: True

ROOT: /content/drive/MyDrive/seminar/UIT-ADrone
DIRS: ['videos', 'test', 'train', 'snippets']
FILES: ['test.json', 'train.json', 'README.txt']
--------------------------------------------------


In [12]:
!ls -l /content/TrafficVision-Transformer/experiments_andt_ADrone_STE_TTE

total 0
lrwxrwxrwx 1 root root 33 Jun  2 15:28 checkpoints -> /content/drive/MyDrive/model_save


In [ ]:
import os

os.chdir('/content/TrafficVision-Transformer')

DATA_DIR = '/content/drive/MyDrive/seminar/UIT-ADrone'

%cd /content/TrafficVision-Transformer

!PYTHONPATH=/content/TrafficVision-Transformer \
python -m scripts.train_ste_tte \
    --train 1 \
    --epochs 2 \
    --batch-size 2 \
    --lr 1e-4 \
    --train-steps 4000 \
    --model-arch b16 \
    --image-size 384 \
    --num-workers 2 \
    --data-dir $DATA_DIR

/content
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/TrafficVision-Transformer/scripts/train_ste_tte.py", line 15, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 3001, in <module>
    _import_device_backends()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2949, in _import_device_backends
    backend_extensions = entry_points(group=group_name)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 913, in entry_points
    return EntryPoints(eps).select(**params)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 911, in <genexpr>
    dist.entry_points for dist in _unique(distributions())
    ^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 471, in entry

In [ ]:
import os

os.chdir('/content/TrafficVision-Transformer')

DATA_DIR = '/content/drive/MyDrive/seminar/UIT-ADrone'

CKPT = '/content/drive/MyDrive/model_save/batch_20000.pth'

%cd /content/TrafficVision-Transformer

!PYTHONPATH=/content/TrafficVision-Transformer \
python -m scripts.train_ste_tte \
    --train 1 \
    --checkpoint-path $CKPT \
    --epochs 1 \
    --batch-size 2 \
    --lr 5e-5 \
    --train-steps 12000 \
    --model-arch b16 \
    --image-size 384 \
    --num-workers 2 \
    --data-dir $DATA_DIR

/content
 *************************************** 
 The experiment name is anomaly_ste_tte 
 *************************************** 
----------------- Config ---------------
                  anomaly_threshold: None                          
                  attn_dropout_rate: 0.0                           
                         batch_size: 2                             
                     checkpoint_dir: experiments/save/anomaly_ste_tte_UIT-ADrone_bs2_lr5e-05_wd0.0001_260602_075417/checkpoints/
                    checkpoint_path: /content/drive/MyDrive/model_save/batch_20000.pth
                           data_dir: /content/drive/MyDrive/seminar/UIT-ADrone
                            dataset: UIT-ADrone                    
                       dropout_rate: 0.1                           
                            emb_dim: 768                           
                             epochs: 1                             
                           exp_name: anomaly_ste_tte  

In [ ]:
!ls /content/drive/MyDrive/model_save

## BƯỚC 8 — Kiểm tra kết quả training

In [13]:
import os

# ===== CHECK CHECKPOINT =====
ckpt_dir = '/content/drive/MyDrive/model_save'

print('=== CHECKPOINT FILES ===')

if os.path.exists(ckpt_dir):

    ckpts = sorted(
        os.listdir(ckpt_dir),
        key=lambda x: int(x.split('_')[1].split('.')[0])
    )

    for f in ckpts:

        size_mb = os.path.getsize(
            os.path.join(ckpt_dir, f)
        ) / 1e6

        print(f'{f:<20} {size_mb:.1f} MB')

    latest_ckpt = ckpts[-1]

    print(f'\n✅ Latest checkpoint: {latest_ckpt}')

else:
    print('❌ Không tìm thấy folder model_save')


# ===== CHECK TRAINING LOG =====
log_path = '/content/TrafficVision-Transformer/training_log_ste_tte.txt'

print('\n=== TRAINING LOG ===')

if os.path.exists(log_path):

    with open(log_path, 'r') as f:

        lines = f.readlines()

        # In 20 dòng cuối
        for line in lines[-20:]:
            print(line.strip())

else:
    print('⚠️ Chưa có training log.')

=== CHECKPOINT FILES ===


ValueError: invalid literal for int() with base 10: 'log'

## BƯỚC 9 — Copy log lên Drive (để không mất sau khi session hết)

In [14]:
import shutil
import os

src = '/content/TrafficVision-Transformer/training_log_ste_tte.txt'

dst = '/content/drive/MyDrive/model_save/training_log_ste_tte.txt'

if os.path.exists(src):

    shutil.copy(src, dst)

    print('✅ Log copied to Drive:')
    print(dst)

else:
    print('❌ Không tìm thấy log file.')

❌ Không tìm thấy log file.


In [15]:
# ===== CHECK FULL PIPELINE BEFORE REAL TEST =====

import os

# ===== PATHS =====
REPO_DIR = '/content/TrafficVision-Transformer'
DATA_DIR = '/content/drive/MyDrive/seminar/UIT-ADrone'
CKPT = '/content/drive/MyDrive/model_save/batch_21000.pth'

print("=== CHECK REPO ===")
print("Repo exists:", os.path.exists(REPO_DIR))

print("\n=== CHECK DATASET ===")
print("Dataset exists:", os.path.exists(DATA_DIR))

print("\n=== CHECK CHECKPOINT ===")
print("Checkpoint exists:", os.path.exists(CKPT))

print("\n=== CHECK TEST SCRIPT ===")
test_script = os.path.join(REPO_DIR, 'scripts/train_ste_tte.py')
print("train_ste_tte.py exists:", os.path.exists(test_script))

print("\n=== CHECK TEST FOLDER ===")
test_frames = os.path.join(DATA_DIR, 'test/frames')
print("Test frames exists:", os.path.exists(test_frames))

print("\n=== CHECK LABELS ===")
test_labels = os.path.join(DATA_DIR, 'test/test_frame_mask')
print("Test labels exists:", os.path.exists(test_labels))

print("\n=== CHECK CHECKPOINT FILE SIZE ===")
if os.path.exists(CKPT):
    size_gb = os.path.getsize(CKPT) / 1e9
    print(f"Checkpoint size: {size_gb:.2f} GB")

print("\n=== CHECK MODIFIED CODE ===")

with open(test_script, 'r') as f:
    content = f.read()

if "path_ckpt = config.checkpoint_path" in content:
    print("✅ checkpoint-path fix detected")
else:
    print("❌ checkpoint-path fix NOT found")

if "training_log_ste_tte.txt" in content:
    print("✅ training log enabled")
else:
    print("⚠️ training log not found")

print("\n=== CHECK SYMLINK ===")

!ls -l /content/TrafficVision-Transformer/experiments_andt_ADrone_STE_TTE

print("\n=== READY STATUS ===")

ready = (
    os.path.exists(REPO_DIR)
    and os.path.exists(DATA_DIR)
    and os.path.exists(CKPT)
    and os.path.exists(test_frames)
    and os.path.exists(test_labels)
)

if ready:
    print("✅ EVERYTHING READY FOR TESTING")
else:
    print("❌ SOMETHING IS MISSING")

=== CHECK REPO ===
Repo exists: True

=== CHECK DATASET ===
Dataset exists: True

=== CHECK CHECKPOINT ===
Checkpoint exists: True

=== CHECK TEST SCRIPT ===
train_ste_tte.py exists: True

=== CHECK TEST FOLDER ===
Test frames exists: True

=== CHECK LABELS ===
Test labels exists: True

=== CHECK CHECKPOINT FILE SIZE ===
Checkpoint size: 1.19 GB

=== CHECK MODIFIED CODE ===
✅ checkpoint-path fix detected
✅ training log enabled

=== CHECK SYMLINK ===
total 0
lrwxrwxrwx 1 root root 33 Jun  2 15:28 checkpoints -> /content/drive/MyDrive/model_save

=== READY STATUS ===
✅ EVERYTHING READY FOR TESTING


In [16]:
import os

os.chdir('/content/TrafficVision-Transformer')

DATA_DIR = '/content/drive/MyDrive/seminar/UIT-ADrone'

CKPT = '/content/drive/MyDrive/model_save/batch_21000.pth'

%cd /content/TrafficVision-Transformer

!PYTHONPATH=/content/TrafficVision-Transformer \
python -m scripts.train_ste_tte \
    --train 0 \
    --checkpoint-path $CKPT \
    --data-dir $DATA_DIR \
    --image-size 384

/content
 *************************************** 
 The experiment name is anomaly_ste_tte 
 *************************************** 
----------------- Config ---------------
                  anomaly_threshold: None                          
                  attn_dropout_rate: 0.0                           
                         batch_size: 8                             
                     checkpoint_dir: experiments/save/anomaly_ste_tte_UIT-ADrone_bs8_lr0.0001_wd0.0001_260602_152946/checkpoints/
                    checkpoint_path: /content/drive/MyDrive/model_save/batch_21000.pth
                           data_dir: /content/drive/MyDrive/seminar/UIT-ADrone
                            dataset: UIT-ADrone                    
                       dropout_rate: 0.1                           
                            emb_dim: 768                           
                             epochs: 5                             
                           exp_name: anomaly_ste_tte 

---
## ✅ Xong Ngày 2!

Sau khi train xong, báo cho nhóm:
- Checkpoint đã lưu trong Google Drive: `experiments_andt_ADrone_STE_TTE/checkpoints/best.pth`
- Training log: `experiments_andt_ADrone_STE_TTE/training_log_ste_tte.txt`

**Ngày 3:** Chạy inference + tính AUC (tao sẽ hướng dẫn sau khi Ngày 2 xong).